Import the dataset and setup git

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Path to dataset files: /kaggle/input/fake-and-real-news-dataset


In [2]:
import os

# List the contents of the downloaded dataset directory
dataset_files = os.listdir(path)
print(f"Files in the dataset directory '{path}':\n{dataset_files}")

Files in the dataset directory '/kaggle/input/fake-and-real-news-dataset':
['True.csv', 'Fake.csv']


Visualize the dataset

In [3]:
import pandas as pd
import os

true_news_df = pd.read_csv(os.path.join(path, 'True.csv'))
fake_news_df = pd.read_csv(os.path.join(path, 'Fake.csv'))

# Load the combined synthetic dataset from the Excel file
synthetic_df = pd.read_excel('synthetic_fake_news_combined.xlsx')
synthetic_df['truth'] = 0
# The original synthetic files had a 'label' column which is now dropped.
# Assuming the combined excel also has a similar column or if not, this drop will be ignored.
if 'label' in synthetic_df.columns:
    synthetic_df = synthetic_df.drop(columns=['label'])

print("First 5 rows of True News dataset:")
display(true_news_df.head())

print("\nFirst 5 rows of Fake News dataset:")
display(fake_news_df.head())

print("\nFirst 5 rows of synthetic dataset:")
display(synthetic_df.head())

First 5 rows of True News dataset:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"



First 5 rows of Fake News dataset:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"



First 5 rows of synthetic dataset:


,title,text,subject,date,truth
0,Global Health Council Approves Mandatory Digit...,The International Health Regulatory Body (IHRB...,worldnews,"February 11, 2017",0
1,World Food Authority Mandates New Certificatio...,According to a report released Wednesday by th...,worldnews,"October 17, 2016",0
2,Historic Border Agreement Signed to Resolve De...,The governments of Arkania and Valoria signed ...,worldnews,"February 22, 2017",0
3,Central Bank of the United Republic Implements...,The Central Bank of the United Republic announ...,worldnews,"September 26, 2017",0
4,NATO Commences Operation Vigilant Kraken Amid ...,NATO forces launched Operation Vigilant Kraken...,worldnews,"September 30, 2017",0


In [4]:

# Add 'truth' column to true_news_df with value 1
true_news_df['truth'] = 1

# Add 'truth' column to fake_news_df with value 0
fake_news_df['truth'] = 0

# Combine the dataframes
combined_df = pd.concat([true_news_df, fake_news_df, synthetic_df], ignore_index=True)

print("Combined DataFrame head:")
display(combined_df.head())

print("\nCombined DataFrame tail:")
display(combined_df.tail())

print(f"\nShape of combined DataFrame: {combined_df.shape}")
print(f"Number of true news entries: {combined_df[combined_df['truth'] == 1].shape[0]}")
print(f"Number of fake news entries: {combined_df[combined_df['truth'] == 0].shape[0]}")

Combined DataFrame head:


,title,text,subject,date,truth
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1



Combined DataFrame tail:


,title,text,subject,date,truth
44920,Southeast Asian Nations Finalize Multi-Billion...,Defense ministers from several Southeast Asian...,worldnews,"May 18, 2017",0
44921,Maritime Task Force Reports Confrontation Duri...,A multi-national maritime task force reported ...,worldnews,"August 01, 2017",0
44922,Budget Negotiators Reach Tentative Agreement o...,Congressional leaders announced a tentative fr...,politicsNews,"May 31, 2017",0
44923,Joint Commission Reports Escalation in Boundar...,"""The situation on the ground has deteriorated ...",worldnews,"January 27, 2016",0
44924,Trans-Continental Energy Pact Signed Amid Conc...,A consortium of four European and Central Asia...,worldnews,"January 15, 2016",0



Shape of combined DataFrame: (44925, 5)
Number of true news entries: 21417
Number of fake news entries: 23508


In [5]:
# 1. Randomize combined_df so rows are in random order
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Create master with only text and truth columns
master = combined_df[["text", "truth"]].copy()

# 3. 70% train / 30% test split
from sklearn.model_selection import train_test_split

train, test = train_test_split(master, test_size=0.3, random_state=42)

print(f"Master shape: {master.shape}")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Master shape: (44925, 2)
Train shape: (31447, 2)
Test shape: (13478, 2)


In [6]:
# Installing Packages

import os
os.environ['KERAS_BACKEND'] = 'tensorflow'
import numpy as np
import matplotlib.pyplot as plt
import keras

keras.utils.set_random_seed(42)

In [7]:
y_train = train['truth'].to_numpy()
y_test = test['truth'].to_numpy()

y_train[:10]

array([0, 0, 1, 0, 0, 1, 0, 0, 0, 1])

In [8]:
# Set the maximum number of tokens
max_tokens = 5000

# Configure the text vectorization layer
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="count",
    ngrams=2,
    standardize=lambda x: x  # Bypasses the string check and leaves text untouched
)

# Let's adapt the Text Vectorization layer using the training corpus
text_vectorization.adapt(train['text'])

# We vectorize our input with the adapted Text Vectorization layer
X_train = text_vectorization(train['text'])
X_test = text_vectorization(test['text'])

pd.DataFrame(X_train, columns = text_vectorization.get_vocabulary())

,[UNK],the,to,of,and,a,in,that,on,s,...,government in,"family,",desperate,backed by,a law,"Of course,",to any,the Paris,that many,said. I
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,192,10,3,5,3,2,7,2,4,4,...,0,0,0,0,0,0,0,0,0,0
2,362,22,11,2,10,8,4,1,2,0,...,0,0,0,0,0,0,0,0,0,0
3,24,4,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,558,21,16,13,15,6,4,5,2,4,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31442,460,25,16,22,7,11,9,4,5,3,...,0,0,0,0,0,0,0,0,0,0
31443,44,1,2,1,2,2,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
31444,671,25,11,14,8,12,8,9,1,8,...,0,0,0,0,0,0,0,0,0,0
31445,199,5,5,4,5,7,7,2,2,4,...,0,0,0,0,0,0,0,0,0,0


In [9]:
inputs = keras.layers.Input(shape=(max_tokens, ))

h1 = keras.layers.Dense(128, activation="relu")(inputs)
h2 = keras.layers.Dense(128, activation="relu")(h1)
h3 = keras.layers.Dense(128, activation="relu")(h2)

outputs = keras.layers.Dense(1, activation="sigmoid")(h3)

bow_model = keras.Model(inputs, outputs)
bow_model.summary()

bow_model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy']
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 673,281 (2.57 MB)

 Trainable params: 673,281 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Fit model on the training data with 20 epochs and batch size of 32
bow_model.fit(
    x=X_train[:200], y=y_train[:200],
    epochs=20, batch_size=32,
    verbose=1,
)

print(f"*** Test accuracy with 200 examples:{bow_model.evaluate(x=X_test, y=y_test)[1]:.2%} ***")


# Fit model on the training data with 20 epochs and batch size of 32
bow_model.fit(
    x=X_train, y=y_train,
    epochs=20, batch_size=32,
    verbose=1,
)

print(f"*** Test accuracy with ALL examples:{bow_model.evaluate(x=X_test, y=y_test)[1]:.2%} ***")

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 307ms/step - accuracy: 0.5600 - loss: 1.2607
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8350 - loss: 0.5249 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9400 - loss: 0.3650 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9600 - loss: 0.2416 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9850 - loss: 0.1184 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0722 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0490 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0340 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0250 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0191 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0151 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.0123 


In [ ]:
example_text = ["In a controversial administrative action expected to disenfranchise roughly 33 million Americans, the Department of Justice announced Tuesday the commencement of the Vital Records Audit Initiative, an effort that will retroactively revoke the citizenship of approximately 10 percent of the US population. Officials claim the sweep targets systemic clerical anomalies in municipal birth registries and naturalization processing between 1975 and 1995, rendering affected individuals temporarily stateless pending a lengthy appeals process. The sudden reclassification strips targeted residents of their passports, federal voting rights, and eligibility for Medicare, while requiring them to apply for provisional biometric work permits to maintain their current employment. Legal experts and civil rights organizations immediately condemned the mandate as a gross violation of the 14th Amendment, with advocacy groups filing a massive class-action injunction in the Ninth Circuit Court of Appeals before the markets even closed. The economic fallout was swift and severe, as major corporate employers warned of an unprecedented labor crisis and the S&P 500 tumbled 4 percent on fears of collapsing consumer spending and mass foreclosures. An emergency joint session of Congress has been convened for Thursday to debate a bipartisan measure that would halt the reclassifications and restore legal parity to the affected demographic."]

vectorized_text = text_vectorization(example_text)

prediction = bow_model.predict(vectorized_text)

#The output is a 2D array (e.g., [[0.85]]). Let's extract and interpret it!
probability = prediction[0][0]

print(f"Raw probability: {probability:.4f}")

#Since you used a sigmoid activation, the output is between 0 and 1
if probability >= 0.5:
    print("Prediction: Class 1 (e.g., Real / Spam)")
else:
    print("Prediction: Class 0 (e.g., Fake / Ham)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step
Raw probability: 0.0011
Prediction: Class 0 (e.g., Fake / Ham)
